# SOLAR: model-output visualisation and synthetic evaluation

This notebook inspects saved P01 model outputs without retraining the computationally intensive conditional KDE. It provides a portfolio-scale overview, a detailed confidence-bound view, and a transparent check against the three supplied synthetic anomaly sets.

In [ ]:
from pathlib import Path
import os

MPL_CONFIG_DIR = Path.cwd() / "tmp" / "matplotlib"
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CONFIG_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
MODEL_PATH = ROOT / "model_outputs" / "1-input" / "P01" / "P01_vs_P10.csv"
SYNTHETIC_DIR = ROOT / "synthetic_anomaly_data"
ASSET_DIR = ROOT / "assets"
ASSET_DIR.mkdir(exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Saved-model overview

In [ ]:
required_columns = {"Actual", "LowerBound", "UpperBound", "Anomaly"}
model_df = pd.read_csv(MODEL_PATH)
missing_columns = required_columns.difference(model_df.columns)
if missing_columns:
    raise KeyError(f"Missing model-output columns: {sorted(missing_columns)}")
if (model_df["LowerBound"] > model_df["UpperBound"]).any():
    raise ValueError("At least one saved lower bound exceeds its upper bound.")

anomaly_count = int(model_df["Anomaly"].sum())
anomaly_rate = model_df["Anomaly"].mean()
print(f"{MODEL_PATH.name}: {len(model_df):,} observations")
print(f"Flags on the original series: {anomaly_count:,} ({anomaly_rate:.3%})")

In [ ]:
anomaly_indices = model_df.index[model_df["Anomaly"].eq(1)]
detail_centre = int(anomaly_indices[0]) if len(anomaly_indices) else len(model_df) // 2
detail_start = max(0, detail_centre - 120)
detail_stop = min(len(model_df), detail_centre + 121)
detail = model_df.iloc[detail_start:detail_stop]

fig, axes = plt.subplots(2, 1, figsize=(13, 7), height_ratios=[1.05, 1.35])
axes[0].plot(model_df.index, model_df["Actual"], color="#255f85", linewidth=0.8, label="Observed P01")
axes[0].scatter(anomaly_indices, model_df.loc[anomaly_indices, "Actual"], color="#c23b22", marker="x", s=24, label="Flagged")
axes[0].set(title=f"Full saved series — {anomaly_count} flags ({anomaly_rate:.2%})", xlabel="15-minute sample index", ylabel="P01 pressure")
axes[0].legend(frameon=True)

axes[1].plot(detail.index, detail["Actual"], color="#255f85", linewidth=1.5, label="Observed P01")
axes[1].fill_between(detail.index, detail["LowerBound"], detail["UpperBound"], color="#5ba66b", alpha=0.28, label="Saved confidence envelope")
detail_flags = detail.index[detail["Anomaly"].eq(1)]
axes[1].scatter(detail_flags, detail.loc[detail_flags, "Actual"], color="#c23b22", marker="x", s=45, label="Flagged")
axes[1].set(title=f"Detailed window around sample {detail_centre:,}", xlabel="15-minute sample index", ylabel="P01 pressure")
axes[1].legend(frameon=True)

fig.suptitle("SOLAR one-input model: P10 → P01", fontweight="bold")
fig.tight_layout()
figure_path = ASSET_DIR / "p01-p10-anomaly-detection.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved {figure_path.relative_to(ROOT)}")

## 2. Synthetic-anomaly evaluation

The supplied anomalies were deliberately constructed to cross the learned limits. Recall therefore measures this controlled test only; it should not be read as evidence of real-world fault coverage. Normal-series flags are counted as false positives in the standard synthetic-label calculation.

In [ ]:
rows = []
for anomaly_path in sorted(SYNTHETIC_DIR.glob("P01_*_Anomalies.csv")):
    anomaly_df = pd.read_csv(anomaly_path)
    if len(anomaly_df) != len(model_df):
        raise ValueError(f"Length mismatch for {anomaly_path.name}.")
    truth = anomaly_df["Anomalous"].astype(bool)
    predicted = (anomaly_df["P01_Modified"] < model_df["LowerBound"]) | (anomaly_df["P01_Modified"] > model_df["UpperBound"])
    true_positive = int((truth & predicted).sum())
    false_positive = int((~truth & predicted).sum())
    false_negative = int((truth & ~predicted).sum())
    precision = true_positive / (true_positive + false_positive)
    recall = true_positive / (true_positive + false_negative)
    f1_score = 2 * precision * recall / (precision + recall)
    anomaly_type = anomaly_path.stem.removeprefix("P01_").removesuffix("_Anomalies")
    rows.append({"Anomaly type": anomaly_type, "Injected points": int(truth.sum()), "Precision": precision, "Recall": recall, "F1": f1_score})

evaluation_df = pd.DataFrame(rows).set_index("Anomaly type")
display(evaluation_df.style.format({"Precision": "{:.3f}", "Recall": "{:.3f}", "F1": "{:.3f}"}))